# Tesla Deliveries & Production Analysis

## End-to-End Machine Learning Pipeline on Sales/Production Data

### Objectives

This project aims to:

- Perform data cleaning and preprocessing
- Conduct exploratory data analysis (EDA)
- Engineer meaningful features
- Build regression models
- Apply cross-validation
- Perform hyperparameter tuning
- Create machine learning pipelines
- Analyze time series patterns
- Forecast future deliveries

Dataset:
Tesla Deliveries and Production Data (2015-2025)

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.model_selection import (
    GridSearchCV,
    TimeSeriesSplit,
    cross_val_score
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

In [ ]:
#Load Dataset
df = pd.read_csv("tesla_deliveries_dataset_2015_2025.csv")

df.head()

## Dataset Overview

Understanding the structure of the dataset before preprocessing.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## Data Cleaning

Checking:
- Missing Values
- Duplicate Rows
- Incorrect Data Types

In [ ]:
df.drop_duplicates(inplace=True)

df.isnull().sum()

In [ ]:
df.columns

## Feature Engineering

Creating new features from existing variables.

In [ ]:
df['Date'] = pd.to_datetime(
    df['Year'].astype(str) +
    '-' +
    df['Month'].astype(str)
)

In [ ]:
df['Quarter'] = df['Date'].dt.quarter

In [ ]:
#Month Name
df['Month_Name'] = df['Date'].dt.month_name()

In [ ]:
#Delivery Efficiency
df['Efficiency'] = (
    df['Estimated_Deliveries']
    /
    df['Production_Units']
)

In [ ]:
#Lag1 Features
df["Lag_1"] = (
    df.groupby("Model")
    ["Estimated_Deliveries"]
    .shift(1)
)

In [ ]:
#lag3
df["Lag_3"] = (
    df.groupby("Model")
    ["Estimated_Deliveries"]
    .shift(3)
)

In [ ]:
df["Rolling_Mean_3"] = (
    df.groupby("Model")
    ["Estimated_Deliveries"]
    .transform(
        lambda x: x.rolling(3).mean()
    )
)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.head()

Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    df["Estimated_Deliveries"],
    kde=True
)

plt.title("Distribution of Estimated Deliveries")
plt.show()

In [ ]:

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


In [ ]:

plt.figure(figsize=(8,5))
df.groupby('Region')['Estimated_Deliveries'].sum().plot(kind='bar')
plt.title('Region-wise Deliveries')
plt.ylabel('Deliveries')
plt.show()


In [ ]:

monthly_sales = df.groupby('Month')['Estimated_Deliveries'].mean()

plt.figure(figsize=(8,5))
monthly_sales.plot(marker='o')
plt.title('Monthly Average Deliveries')
plt.ylabel('Average Deliveries')
plt.show()


In [ ]:

plt.figure(figsize=(8,5))
sns.boxplot(x='Model', y='Avg_Price_USD', data=df)
plt.title('Model-wise Average Price')
plt.show()


In [ ]:

plt.figure(figsize=(8,5))
sns.scatterplot(
    x='Battery_Capacity_kWh',
    y='Range_km',
    hue='Model',
    data=df
)
plt.title('Battery Capacity vs Range')
plt.show()


In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x=df["Estimated_Deliveries"]
)

plt.title("Outlier Detection")
plt.show()

In [ ]:
region_sales = (
    df.groupby("Region")
    ["Estimated_Deliveries"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10,5))

sns.barplot(
    x=region_sales.index,
    y=region_sales.values
)

plt.title("Total Deliveries by Region")
plt.xticks(rotation=45)

plt.show()

In [ ]:
model_sales = (
    df.groupby("Model")
    ["Estimated_Deliveries"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10,5))

sns.barplot(
    x=model_sales.index,
    y=model_sales.values
)

plt.title("Total Deliveries by Model")
plt.xticks(rotation=45)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="Production_Units",
    y="Estimated_Deliveries"
)

plt.title("Production vs Deliveries")
plt.show()

# Time Series Analysis

To analyze delivery trends over time, deliveries are aggregated at the monthly level.

This prevents duplicate dates from creating misleading visualizations.

In [ ]:
monthly_delivery = (
    df.groupby("Date")
    ["Estimated_Deliveries"]
    .sum()
    .reset_index()
)

In [ ]:
plt.figure(figsize=(12,5))

sns.lineplot(
    data=monthly_delivery,
    x="Date",
    y="Estimated_Deliveries"
)

plt.title("Monthly Tesla Deliveries Trend")

plt.show()

In [ ]:
monthly_delivery["Rolling_Mean"] = (
    monthly_delivery["Estimated_Deliveries"]
    .rolling(12)
    .mean()
)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_delivery["Date"],
    monthly_delivery["Estimated_Deliveries"],
    label="Original"
)

plt.plot(
    monthly_delivery["Date"],
    monthly_delivery["Rolling_Mean"],
    label="Rolling Mean"
)

plt.legend()

plt.title("Monthly Trend with Rolling Mean")

plt.show()

# Machine Learning Preparation

Preparing features and target variable for predictive modeling.

In [ ]:
features = [
    'Production_Units',
    'Avg_Price_USD',
    'Battery_Capacity_kWh',
    'Range_km',
    'CO2_Saved_tons',
    'Charging_Stations',
    'Quarter',
    'Efficiency',
    'Lag_1',
    'Lag_3',
    'Rolling_Mean_3',
    'Region',
    'Model',
    'Source_Type'
]

target = "Estimated_Deliveries"

In [ ]:
X = df[features]
y = df[target]

In [ ]:
categorical_features = [
    'Region',
    'Model',
    'Source_Type'
]

numeric_features = [
    col for col in features
    if col not in categorical_features
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [ ]:
split_index = int(len(df)*0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [ ]:
#(Linear Regression)
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train,y_train)


In [ ]:
pred_lr = linear_model.predict(X_test)

In [ ]:
#Ridge
ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

ridge_model.fit(X_train,y_train)



In [ ]:
pred_ridge = ridge_model.predict(X_test)

In [ ]:
#lasso
lasso_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso())
])

lasso_model.fit(X_train,y_train)

pred_lasso = lasso_model.predict(X_test)

In [ ]:
#(Random Forest)
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42
    ))
])

rf_model.fit(X_train,y_train)

pred_rf = rf_model.predict(X_test)

In [ ]:
def evaluate(y_true,y_pred):

    rmse = np.sqrt(
        mean_squared_error(y_true,y_pred)
    )

    mae = mean_absolute_error(
        y_true,y_pred
    )

    r2 = r2_score(
        y_true,y_pred
    )

    return rmse,mae,r2

In [ ]:
results = pd.DataFrame({
    "Model":[
        "Linear Regression",
        "Ridge",
        "Lasso",
        "Random Forest"
    ],
    "RMSE":[
        evaluate(y_test,pred_lr)[0],
        evaluate(y_test,pred_ridge)[0],
        evaluate(y_test,pred_lasso)[0],
        evaluate(y_test,pred_rf)[0]
    ],
    "MAE":[
        evaluate(y_test,pred_lr)[1],
        evaluate(y_test,pred_ridge)[1],
        evaluate(y_test,pred_lasso)[1],
        evaluate(y_test,pred_rf)[1]
    ],
    "R2":[
        evaluate(y_test,pred_lr)[2],
        evaluate(y_test,pred_ridge)[2],
        evaluate(y_test,pred_lasso)[2],
        evaluate(y_test,pred_rf)[2]
    ]
})

results.sort_values(
    "R2",
    ascending=False
)

(Hyperparameter Tuning)

In [ ]:
param_grid = {
    "model__n_estimators":[100,200],
    "model__max_depth":[5,10,None]
}

grid = GridSearchCV(
    rf_model,
    param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train,y_train)

grid.best_params_
print("Best Score:", grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

predictions = best_model.predict(X_test)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    y_test.values,
    label="Actual"
)

plt.plot(
    predictions,
    label="Predicted"
)

plt.legend()

plt.title(
    "Actual vs Predicted Deliveries"
)

plt.show()

# Conclusion

The project successfully developed a complete Machine Learning pipeline for Tesla delivery prediction.

Key achievements:

- Data Cleaning
- Exploratory Data Analysis
- Feature Engineering
- Time Series Trend Analysis
- Machine Learning Pipelines
- Linear Regression
- Ridge Regression
- Lasso Regression
- Random Forest Regression
- Hyperparameter Tuning
- Model Evaluation

Among all models, the best-performing model was selected using RMSE, MAE, and R² metrics.

The analysis demonstrates how production, infrastructure, and historical patterns influence Tesla deliveries.